In [11]:
# Persiapan ulang untuk latihan
import os, sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder.appName("Latihan7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv", header=True, inferSchema=True)
df_toko = spark.read.json("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_produk.csv", header=True, inferSchema=True)
df_gabungan = df_transaksi.join(df_toko, on="toko_id").join(df_produk, on="product_id")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
print("Siap untuk latihan. Total baris:", df_gabungan.count())

Siap untuk latihan. Total baris: 5000


**Soal 1.** Menggunakan Spark SQL, tampilkan **kategori produk** dengan total `qty` (jumlah unit terjual, bukan nilai rupiah) tertinggi secara keseluruhan.

In [12]:
# Jawaban Soal 1: Menampilkan kategori produk dengan total qty tertinggi menggunakan Spark SQL
# Mendaftarkan DataFrame gabungan sebagai Temporary View SQL
df_gabungan.createOrReplaceTempView("gabungan_latihan")

# Mengeksekusi kueri SQL untuk mengagregasi total kuantitas per kategori
soal1_kategori_qty = spark.sql('''
    SELECT 
        kategori, 
        SUM(qty) AS total_qty
    FROM gabungan_latihan
    GROUP BY kategori
    ORDER BY total_qty DESC
''')

print("=== Kategori Produk Berdasarkan Total Qty Terjual (Spark SQL) ===")
soal1_kategori_qty.show()

=== Kategori Produk Berdasarkan Total Qty Terjual (Spark SQL) ===


+----------+---------+
|  kategori|total_qty|
+----------+---------+
|Elektronik|     6736|
|   Fashion|     5307|
|   Makanan|     3114|
+----------+---------+



**Soal 2.** Menggunakan window function, tentukan **toko dengan penjualan tertinggi** untuk **masing-masing kategori produk** (top-1 saja, gunakan `row_number()`).

In [13]:
# Jawaban Soal 2: Menentukan toko dengan penjualan tertinggi untuk masing-masing kategori produk (Top-1 Window Function)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, sum as spark_sum

# Langkah 1: Meringkas total penjualan per kombinasi kategori dan nama toko
ringkasan_kategori_toko = df_gabungan.groupBy("kategori", "nama_toko").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Langkah 2: Menentukan spesifikasi Window berdasarkan partisi 'kategori' dan urutan 'total_penjualan' menurun
window_kategori = Window.partitionBy("kategori").orderBy(col("total_penjualan").desc())

# Langkah 3: Menerapkan fungsi analitis row_number() dan memfilter hanya peringkat 1 (top-1)
top1_toko_per_kategori = ringkasan_kategori_toko.withColumn("peringkat", row_number().over(window_kategori)) \
    .filter(col("peringkat") == 1) \
    .drop("peringkat") \
    .orderBy(col("total_penjualan").desc())

print("=== Toko dengan Penjualan Tertinggi untuk Masing-Masing Kategori (Top-1 Window Function) ===")
top1_toko_per_kategori.show()

=== Toko dengan Penjualan Tertinggi untuk Masing-Masing Kategori (Top-1 Window Function) ===


+----------+-------------+---------------+
|  kategori|    nama_toko|total_penjualan|
+----------+-------------+---------------+
|Elektronik|Toko Magelang|      202150000|
|   Fashion|Toko Semarang|      123000000|
|   Makanan|Toko Magelang|      108200000|
+----------+-------------+---------------+



**Soal 3.** Simpan hasil `ringkasan_eksekutif` (dari Sub-bab 7.4 di atas — silakan buat ulang) ke HDFS sebagai Parquet **tanpa partisi**, pada path `/user/mahasiswa/latihan7/ringkasan_eksekutif`.

In [14]:
# Jawaban Soal 3: Menyimpan ringkasan eksekutif ke HDFS sebagai Parquet tanpa partisi
from pyspark.sql.functions import avg, count

# Menyusun ulang ringkasan eksekutif dari df_gabungan
ringkasan_eksekutif_latihan = df_gabungan.groupBy("nama_toko", "kota", "manager").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

# Path target HDFS (tanpa partisi)
path_soal3_hdfs = "hdfs://localhost:9000/user/mahasiswa/latihan7/ringkasan_eksekutif"

# Menyimpan DataFrame ke format Parquet tanpa pemanggilan .partitionBy()
ringkasan_eksekutif_latihan.write.mode("overwrite").parquet(path_soal3_hdfs)
print(f"Hasil ringkasan eksekutif berhasil disimpan ke HDFS: {path_soal3_hdfs}\n")

# Verifikasi berkas di HDFS menggunakan perintah shell
print("=== Struktur Berkas di HDFS ===")
!hdfs dfs -ls /user/mahasiswa/latihan7/ringkasan_eksekutif

# Verifikasi dengan membaca kembali berkas Parquet dari HDFS
df_verifikasi_soal3 = spark.read.parquet(path_soal3_hdfs)
print("\n=== Verifikasi Data Dibaca Kembali dari HDFS ===")
print("Jumlah baris data:", df_verifikasi_soal3.count())
df_verifikasi_soal3.show()

Hasil ringkasan eksekutif berhasil disimpan ke HDFS: hdfs://localhost:9000/user/mahasiswa/latihan7/ringkasan_eksekutif

=== Struktur Berkas di HDFS ===


Found 2 items
-rw-r--r--   3 raka supergroup          0 2026-10-03 11:25 /user/mahasiswa/latihan7/ringkasan_eksekutif/_SUCCESS
-rw-r--r--   3 raka supergroup       2087 2026-10-03 11:25 /user/mahasiswa/latihan7/ringkasan_eksekutif/part-00000-0441db95-0534-44f5-8ddb-03632b644e38-c000.snappy.parquet



=== Verifikasi Data Dibaca Kembali dari HDFS ===


Jumlah baris data: 4


+---------------+----------+-------+------------------------+----------------+-------------------------+
|      nama_toko|      kota|manager|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------------+----------+-------+------------------------+----------------+-------------------------+
|  Toko Semarang|  Semarang|   Rani|               416250000|            1306|       318721.28637059726|
|  Toko Magelang|  Magelang|   Bayu|               408550000|            1315|       310684.41064638784|
|Toko Yogyakarta|Yogyakarta|   Bayu|               389150000|            1200|        324291.6666666667|
|      Toko Solo|      Solo|   Rani|               369950000|            1179|        313782.8668363019|
+---------------+----------+-------+------------------------+----------------+-------------------------+



**Soal 4 (Refleksi menyeluruh).** Sebagai persiapan UTS, jelaskan dalam 4-5 kalimat: bagaimana **HDFS, Spark, dan format Parquet saling melengkapi** dalam satu ekosistem Big Data? Gunakan istilah-istilah yang telah anda pelajari sejak Pertemuan 3 (mis. distributed storage, in-memory processing, columnar format, partitioning).

**Jawaban:**
HDFS, Apache Spark, dan format Parquet membentuk fondasi arsitektur Big Data modern yang saling melengkapi secara optimal dalam hal kapasitas penyimpanan, performa komputasi, dan efisiensi I/O. **HDFS** menyediakan lapisan *distributed storage* yang andal, toleran terhadap kegagalan (*fault-tolerant*), serta *scalable* untuk menampung volume data mentah berukuran masif di seluruh klaster mesin komoditas. **Apache Spark** berperan sebagai *distributed processing engine* berbasis *in-memory processing*, yang mampu mengeksekusi transformasi kompleks (join, agregasi, window function) puluhan kali lebih cepat dibanding MapReduce tradisional tanpa latensi baca/tulis disk berulang. Sementara itu, **Parquet** menyempurnakan ekosistem ini sebagai *columnar format* dengan rasio kompresi tinggi (Snappy) serta mendukung *partitioning* dan *predicate pushdown*, sehingga Spark hanya perlu membaca kolom dan blok data yang benar-benar relevan langsung dari blok HDFS. Sinergi ketiganya menciptakan pipeline data end-to-end yang hemat sumber daya penyimpanan, minim kemacetan I/O jaringan, dan memberikan *throughput* analisis data skala besar yang sangat tinggi.

In [15]:
# Menutup sesi Spark latihan sebelum masuk ke Tugas Mandiri
spark.stop()
print("SparkSession Latihan7 berhasil ditutup.")

SparkSession Latihan7 berhasil ditutup.
